In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardt(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.12681050598621368
epoch:  1, loss: 0.12405683845281601
epoch:  2, loss: 0.12137742340564728
epoch:  3, loss: 0.11820472776889801
epoch:  4, loss: 0.11492534726858139
epoch:  5, loss: 0.11168256402015686
epoch:  6, loss: 0.10836968570947647
epoch:  7, loss: 0.10510502010583878
epoch:  8, loss: 0.1018313467502594
epoch:  9, loss: 0.09863277524709702
epoch:  10, loss: 0.09542033821344376
epoch:  11, loss: 0.09281736612319946
epoch:  12, loss: 0.09023503214120865
epoch:  13, loss: 0.08818336576223373
epoch:  14, loss: 0.08499260991811752
epoch:  15, loss: 0.08188941329717636
epoch:  16, loss: 0.0789741799235344
epoch:  17, loss: 0.07637380808591843
epoch:  18, loss: 0.07357944548130035
epoch:  19, loss: 0.07089774310588837
epoch:  20, loss: 0.06827714294195175
epoch:  21, loss: 0.06575069576501846
epoch:  22, loss: 0.06324704736471176
epoch:  23, loss: 0.06083224341273308
epoch:  24, loss: 0.05840352177619934
epoch:  25, loss: 0.056050729006528854
epoch:  26, loss: 0.053

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9847433006488127
Test metrics:  R2 = 0.8506944678793599


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.47720471024513245
epoch:  1, loss: 0.24334222078323364
epoch:  2, loss: 0.05015361309051514
epoch:  3, loss: 0.038613881915807724
epoch:  4, loss: 0.032603610306978226
epoch:  5, loss: 0.026981210336089134
epoch:  6, loss: 0.024639319628477097
epoch:  7, loss: 0.015076872892677784
epoch:  8, loss: 0.014191432856023312
epoch:  9, loss: 0.012685867957770824
epoch:  10, loss: 0.012631107121706009
epoch:  11, loss: 0.012339743785560131
epoch:  12, loss: 0.011754697188735008
epoch:  13, loss: 0.011716957204043865
epoch:  14, loss: 0.011390776373445988
epoch:  15, loss: 0.01096494123339653
epoch:  16, loss: 0.010932769626379013
epoch:  17, loss: 0.010687720030546188
epoch:  18, loss: 0.01045706495642662
epoch:  19, loss: 0.01023607887327671
epoch:  20, loss: 0.01006318349391222
epoch:  21, loss: 0.009962925687432289
epoch:  22, loss: 0.009948932565748692
epoch:  23, loss: 0.009892899543046951
epoch:  24, loss: 0.00975966639816761
epoch:  25, loss: 0.009644996374845505
epoc

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9958467167385524
Test metrics:  R2 = 0.9829229724016273
